# Settings

In [1]:
from pathlib import Path

import vine_data as vd

H5_PATH = vd.DEFAULT_H5_PATH

# Load Labels

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Open read-only; keep the handle for later cells
f = vd.open_h5(H5_PATH)
wavelengths = vd.load_wavelengths(f)

print("\nroot attrs:")
for k, v in f.attrs.items():
    print(f"  {k}: {v}")
print(f"\nwavelengths: {wavelengths.shape[0]} bands, "
      f"{wavelengths.min():.1f}-{wavelengths.max():.1f} nm")

meta = vd.index_vines(f)
print(f"{len(meta)} vines indexed")
display(meta.head(5))


root attrs:
  class_names: ['Negative' 'Positive']
  default_bands: [70 53 19]
  default_bands_note: R, G, B band numbers from the Specim IQ header, 1-based (ENVI convention)
  label_names: ['red_blotch' 'leafroll3']
  label_year: 2026
  layout: cube is (lines, bands, samples) as stored in the ENVI BIL files
  sensor: Specim IQ
  sensor_docs: docs/Specim-IQ-Datasheet.pdf

wavelengths: 204 bands, 397.3-1003.6 nm


998 vines indexed


,path,variety,row,vine,ccovi_code,capture_date,red_blotch,leafroll3,label,cube_shape,dtype
0,Baco/R1/V1,Baco,R1,1,B01,2026-08-12,1,1,"[1, 1]","(151, 204, 157)",float32
1,Baco/R1/V2,Baco,R1,2,B02,2026-08-12,1,1,"[1, 1]","(158, 204, 151)",float32
2,Baco/R1/V3,Baco,R1,3,B03,2026-08-12,1,0,"[1, 0]","(141, 204, 150)",float32
3,Baco/R1/V4,Baco,R1,4,B04,2026-08-12,1,0,"[1, 0]","(158, 204, 153)",float32
4,Baco/R1/V5,Baco,R1,5,B05,2026-08-12,0,1,"[0, 1]","(140, 204, 171)",float32


# Cube sizes

In [3]:
# cube_shape is (lines, bands, samples). Bands are always 204; size is the spatial cut-out.
shapes = np.asarray(meta["cube_shape"].to_list(), dtype=np.int64)
lines, bands, samples = shapes[:, 0], shapes[:, 1], shapes[:, 2]
n_spatial = lines * samples

def extrema(values, name):
    i_min, i_max = int(values.argmin()), int(values.argmax())
    print(f"{name}: min {values[i_min]} ({meta.loc[i_min, 'path']}), "
          f"max {values[i_max]} ({meta.loc[i_max, 'path']})")

print(f"{len(meta)} vines")
print("cube layout: (lines, bands, samples)")
extrema(lines, "lines (height)")
extrema(samples, "samples (width)")
extrema(bands, "bands")
extrema(n_spatial, "spatial pixels (lines x samples)")

998 vines
cube layout: (lines, bands, samples)
lines (height): min 32 (Cab Franc/R4/V5), max 211 (Baco/R1/V44)
samples (width): min 42 (Cab Franc/R4/V5), max 204 (Baco/R5/V22)
bands: min 204 (Baco/R1/V1), max 204 (Baco/R1/V1)
spatial pixels (lines x samples): min 1344 (Cab Franc/R4/V5), max 37611 (Baco/R4/V22)


In [4]:
# First vine's cube lives at f[path]["cube"] — (lines, bands, samples) = (height, 204, width).
# Values are reflectance (float32), typically 0–1; >1.0 is a specular glint.
row = meta.iloc[0]
print(row[["path", "variety", "cube_shape", "red_blotch", "leafroll3"]])

cube = f[row["path"]]["cube"]
print(f"\ndataset: {cube}")
print(f"shape {cube.shape}  dtype {cube.dtype}")
print(f"min {cube[:].min():.4f}  mean {cube[:].mean():.4f}  max {cube[:].max():.4f}")

h, n_bands, w = cube.shape
y, x = h // 2, w // 2
spectrum = cube[y, :, x]  # one pixel, all 204 bands
print(f"\ncentre pixel (line={y}, sample={x}):")
print(f"  min/mean/max {spectrum.min():.4f} / {spectrum.mean():.4f} / {spectrum.max():.4f}")
print("  first 12 bands:")
for nm, v in zip(wavelengths[:12], spectrum[:12]):
    print(f"    {nm:7.1f} nm   {v:.6f}")

band_i = int(np.argmin(np.abs(wavelengths - 550)))
patch = np.asarray(cube[y - 2:y + 3, band_i, x - 2:x + 3]) 
print(f"\n5x5 patch at {wavelengths[band_i]:.1f} nm (band {band_i}):")
np.set_printoptions(precision=4, suppress=True)
print(patch)

path               Baco/R1/V1
variety                  Baco
cube_shape    (151, 204, 157)
red_blotch                  1
leafroll3                   1
Name: 0, dtype: object

dataset: <HDF5 dataset "cube": shape (151, 204, 157), type "<f4">
shape (151, 204, 157)  dtype float32
min 0.0000  mean 0.3632  max 1.3488

centre pixel (line=75, sample=78):
  min/mean/max 0.1243 / 0.6311 / 1.1455
  first 12 bands:
      397.3 nm   0.730769
      400.2 nm   0.606061
      403.1 nm   0.538462
      406.0 nm   0.446809
      408.9 nm   0.403509
      411.7 nm   0.348485
      414.6 nm   0.333333
      417.5 nm   0.301205
      420.4 nm   0.282609
      423.3 nm   0.287129
      426.2 nm   0.256637
      429.1 nm   0.256000

5x5 patch at 548.5 nm (band 52):
[[0.2874 0.2842 0.2777 0.2659 0.2896]
 [0.2968 0.2871 0.2828 0.2688 0.2817]
 [0.3194 0.3118 0.2978 0.2774 0.2742]
 [0.2989 0.3075 0.3054 0.2882 0.2763]
 [0.2849 0.2785 0.2774 0.271  0.2774]]


In [5]:
def fmt(count, n):
    return f"{int(count)} ({100 * count / n:.1f}%)".replace(".0%", "%")

# Mutually exclusive status: RB only / LR only / both / healthy
rows_status = []
for variety, g in meta.groupby("variety", sort=False):
    n = len(g)
    rb_only = ((g.red_blotch == 1) & (g.leafroll3 == 0)).sum()
    lr_only = ((g.red_blotch == 0) & (g.leafroll3 == 1)).sum()
    both = ((g.red_blotch == 1) & (g.leafroll3 == 1)).sum()
    healthy = ((g.red_blotch == 0) & (g.leafroll3 == 0)).sum()
    rows_status.append({
        "Red Blotch": fmt(rb_only, n),
        "Leafroll": fmt(lr_only, n),
        "Both": fmt(both, n),
        "Healthy": fmt(healthy, n),
    })

status_distribution = pd.DataFrame(
    rows_status, index=list(meta.groupby("variety", sort=False).groups.keys())
)
status_distribution

,Red Blotch,Leafroll,Both,Healthy
Baco,82 (33.1%),54 (21.8%),53 (21.4%),59 (23.8%)
Cab Franc,240 (96%),0 (0%),1 (0.4%),9 (3.6%)
Chardonnay,38 (15.2%),72 (28.8%),50 (20%),90 (36%)
Vidal,157 (62.8%),13 (5.2%),12 (4.8%),68 (27.2%)
